# Week6_ex5 - Electromagnet Attracting a Permanent Magnet

A coil-driven electromagnet ("bar" core + coil) facing a permanent N42 magnet -- sweeps the coil current and logs force/torque on the magnet at each step to build a force-vs-current table.

Note: the original notebook computed a single fixed current and its report already referenced a swept "a" (current) variable that was registered but never actually swept anywhere. This version adds the missing parametric sweep so the report matches what it's asking for.

In [ ]:
import pandas as pd
import numpy as np
import os
import ansys.aedt.core
import math
import shutil
import time

In [ ]:
## Variables ##

gap = 5      # gap between the magnet and the electromagnet
Width = 20   # width
Height = 10  # height
Depth = 200  # depth

coil_w = 5   # coil thickness

J = 10       # current density

In [ ]:
# 1. Launch AEDT
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)

# 2. Turn off autosave
DT.disable_autosave()

# 3. Create project and set solution type
sol_type = "Magnetostatic"
M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type)

# 4. odesign object for script-recording style calls
oDesign = M3D.odesign

In [ ]:
## Set up output directory for results ##

proj_name = "Week6_ex5"   # matches this exercise's folder name

# If ANSYS_PROJECT_DIR is set on this machine, save there.
# Otherwise, fall back to the notebook's own working directory,
# so this stays portable for anyone else running the notebook as-is.
base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
dir = os.path.join(base_dir, proj_name)
print(dir)
os.makedirs(dir, exist_ok=True)

desi_name = "Week6_ex5"

In [ ]:
## Save project and apply design name ##

proj = M3D.oproject
proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)
M3D.rename_design(desi_name, save=False)

# Save again so the design-name change is committed to disk
M3D.save_project()

In [ ]:
## Model units ##

M3D.modeler.model_units = "mm"

In [ ]:
## Draw the magnet ##

# add the N42 magnet material if it isn't already in the library
if not M3D.materials.exists_material("N42"):
   M3D.materials.add_material("N42")
N42 = M3D.materials["N42"]
N42.permeability.value = 1.05


# magnet
origin = [-Width/2, -(gap/2 + Height), -Depth/2]   # separated from the electromagnet by gap
sizes = [Width, Height, Depth]
magnet1 = M3D.modeler.create_box(origin=origin, sizes=sizes, name="magnet1", material="N42")

In [ ]:
## Magnetize the magnet ##

# local coordinate system attached to the magnet's face
origin = magnet1.faces[0].center
M3D.modeler.create_coordinate_system(origin=origin, reference_cs='Global', name="FaceCS1", mode='axis', view='iso', x_pointing=None, y_pointing=None, psi=0, theta=0, phi=0, u=None)

# reorient the magnet to that face coordinate system
magnet1.part_coordinate_system = "FaceCS1"
M3D.modeler.set_working_coordinate_system("Global") # back to the global coordinate system

# check and set the magnetization direction
N42 = M3D.materials.exists_material(material="N42")
print(N42.get_magnetic_coercivity())

N42.set_magnetic_coercivity('970116', x=0, y=-1, z=0)   # coercivity is positive, so the magnetization direction is reversed (= +Y direction)
print(N42.get_magnetic_coercivity())

In [ ]:
## Draw the electromagnet ##

origin = [-Width/2 - coil_w, gap/2, -Depth/2 - coil_w]   # coil starting corner
sizes = [Width + coil_w*2, Height, Depth + coil_w*2]
coil = M3D.modeler.create_box(origin=origin, sizes=sizes, name="coil", material="copper")

origin = [-Width/2, gap/2, -Depth/2]
sizes = [Width, Height, Depth]
bar = M3D.modeler.create_box(origin=origin, sizes=sizes, name="bar", material="air")

# hollow out the coil with the bar-shaped cavity
coil.subtract(tool_list=[bar], keep_originals=True)

In [ ]:
## Coil terminal and current ##

M3D.modeler.section(assignment=coil, plane="YZ", create_new=True, section_cross_object=False)
coil_section = M3D.modeler.sheet_objects[-1]    # the most recently created section object

coil_terminal = M3D.modeler.separate_bodies(assignment=coil_section, create_group=False)   # split the section into its own object
M3D.modeler.delete(assignment=(M3D.modeler.sheet_objects[-1]))    # delete the leftover cross-section from the split

terminal_area = coil_terminal[0].faces[0].area   # coil terminal area (mm^2)
current = -J * terminal_area                     # current from the current density (A)
M3D["a"] = f"{current}A"                          # register the computed current as a Maxwell design variable, with an explicit unit

# apply the current
current1 = M3D.assign_current(assignment=coil_terminal[0], amplitude="a", name="Current1")

In [ ]:
# quick check of the computed terminal area and nominal current
print(terminal_area, current)

In [ ]:
M3D.assign_force(assignment=magnet1, coordinate_system='Global', is_virtual=True, force_name="Force1")

# torque about X
M3D.assign_torque(assignment=magnet1, coordinate_system="Global", axis="X", is_virtual=True, torque_name="Torque_X")

# torque about Y
M3D.assign_torque(assignment=magnet1, coordinate_system="Global", axis="Y", is_virtual=True, torque_name="Torque_Y")

# torque about Z
M3D.assign_torque(assignment=magnet1, coordinate_system="Global", axis="Z", is_virtual=True, torque_name="Torque_Z")

In [ ]:
## Region ##

region = M3D.modeler.create_region(pad_value=[1000, 1000, 1000, 1000, 100, 100], pad_type='Percentage Offset', name='Region')

In [ ]:
## Analyze Setup ##

# setup object
my_setup = M3D.create_setup(name="Setup1")

# check the analysis setup properties for the current solution type --
# they're stored as a dictionary on the setup object's props attribute
display(my_setup.props)

In [ ]:
# adjust whichever of the props displayed above is needed
# MaximumPasses: controls solve accuracy through repeated mesh refinement passes
my_setup.props['MaximumPasses'] = 25

In [ ]:
## Sweep over coil current (Python loop, not Optimetrics) ##

# NOTE: switched from M3D.parametrics.add(...) to a plain Python loop -- same fix
# as Week6_ex4's Gap sweep. Optimetrics sweeps don't automatically retain scalar
# Force/Torque output-variable data per variation (needs a separate Expression
# Cache setup that assign_force()/assign_torque() don't configure on their own),
# so neither the GUI Data Table/Rectangular Plot report nor get_solution_data()
# could pull Force1.Force_mag/etc. across the swept points ("No trace data for
# quantities..."). Re-solving one current value at a time and reading the result
# immediately after each solve sidesteps this.

current_values = np.linspace(0, current, 11)  # 11 points, 0 up to the nominal current

rows = []
for a_val in current_values:
    M3D["a"] = f"{a_val}A"
    my_setup.analyze()

    data = M3D.post.get_solution_data(
        expressions=["Force1.Force_mag", "Force1.Force_x", "Force1.Force_y", "Force1.Force_z"],
        setup_sweep_name="Setup1 : LastAdaptive",
    )

    rows.append({
        "current_A": a_val,
        "Force_mag_N": data.data_real("Force1.Force_mag")[0],
        "Force_x_N": data.data_real("Force1.Force_x")[0],
        "Force_y_N": data.data_real("Force1.Force_y")[0],
        "Force_z_N": data.data_real("Force1.Force_z")[0],
    })
    print(rows[-1])

df = pd.DataFrame(rows)

csv_name = "Week6_ex5_force.csv"
csv_path = csv_name  # saved flat, next to the notebook -- same convention as Images/
df.to_csv(csv_path, index=False)

df.head()

In [ ]:
## Save project ##

M3D.save_project()